In [ ]:
!chmod +x install.sh
!./install.sh

In [ ]:
!git clone https://github.com/facebookresearch/sam3.git
!cd sam3
!pip install -e .
!pip install decord psutil
!cd ..

In [1]:
from huggingface_hub import login
login(token="token")

# Imports

In [2]:
from gradio_utils import *
from utils import *

Task was destroyed but it is pending!
task: <Task pending name='Task-31' coro=<_async_in_context.<locals>.run_in_context() done, defined at /venv/main/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-32' coro=<Kernel.shell_main() running at /venv/main/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /venv/main/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/venv/main/lib/python3.11/site-packages/gradio/_simple_templates/simpledropdown.py:7: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  from gradio.components.base import Component, FormComponent
Task was destroyed but it is pending!
task: <Task pending name='Task-32' coro=<Kernel.shell_main() running at /venv/main/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>
/test/sam-3d-body/sam_3d_body/models/heads/mhr_head.py:33: UserWarning: Momentum is no

# Model loading (skeleton visualizer & pose estimator)

In [4]:
LIGHT_BLUE = (0.65098039, 0.74117647, 0.85882353)

skeleton_visualizer = SkeletonVisualizer(line_width=2, radius=5)
skeleton_visualizer.set_pose_meta(mhr70_pose_info)

estimator = setup_sam_3d_body(hf_repo_id="facebook/sam-3d-body-dinov3")

Loading SAM 3D Body model from facebook/sam-3d-body-dinov3...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading SAM 3D Body model...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov3_main
Ignored kwargs: {'drop_path': 0.1}
The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.encoder.mask_token, head_pose.hand_pose_comps_ori, head_pose.mhr.face_expressions_model.shape_vectors, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_indices, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_weight, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.2.weight, head_pose.mhr.character_torch.skeleton.joint_translation_offsets, head_pose.mhr.character_torch.skeleton.joint_prerotations, head_pose.mhr.character_torch.skeleton.pmi, head_pose.mhr.character_torch.skeleton.joint_parents, head_pose.mhr.character_torch.mesh.rest_vertices, head_pose.mhr.character_torch.mesh.faces, head_pose.mhr.character_torch.mesh.texcoords, head_pose.mhr.character_torch.mesh.texcoord_faces, head_pose.mhr.character_torch.parameter_transform.parameter_tra

Loading human detector from vitdet...
########### Using human detector: ViTDet...


ModuleNotFoundError: No module named 'pkg_resources'

# Gradio

In [ ]:
with gr.Blocks(title="SAM 3D Body — Soccer Analysis") as demo:

    gr.Markdown("# SAM 3D Body — Soccer Biomechanics Prototype")

    # ── Tab 1: Single image ──────────────────────────────────────────────────
    with gr.Tab("Single Image"):
        gr.Markdown("Upload an image to reconstruct the 3D body mesh and compute biomechanical metrics.")
        with gr.Row():
            with gr.Column():
                img_input   = gr.Image(label="Input Image", type="numpy")
                img_btn     = gr.Button("Run", variant="primary")
            with gr.Column():
                img_output  = gr.Image(label="Reconstruction")
                img_metrics = gr.Textbox(label="Metrics", lines=10)
        img_btn.click(run_process_image, inputs=img_input, outputs=[img_output, img_metrics])

    # ── Tab 2: Video — all players ───────────────────────────────────────────
    with gr.Tab("Video — All Players"):
        gr.Markdown("Upload a video to render the 3D mesh for all detected players.")
        with gr.Row():
            with gr.Column():
                vid_mesh_input  = gr.Video(label="Input Video")
                vid_mesh_btn    = gr.Button("Run", variant="primary")
            with gr.Column():
                vid_mesh_output = gr.Video(label="Output Video")
                vid_mesh_status = gr.Textbox(label="Status")
        vid_mesh_btn.click(
            run_process_video_mesh,
            inputs=vid_mesh_input,
            outputs=[vid_mesh_output, vid_mesh_status]
        )

    # ── Tab 3: Video — track selected player ─────────────────────────────────
    with gr.Tab("Video — Track Player"):
        gr.Markdown(
            "**Step 1** — Upload a video and click **Preview** to detect players on frame 0.  \n"
            "**Step 2** — Enter the number shown on the player you want to track.  \n"
            "**Step 3** — Click **Track** to process the full video."
        )
        with gr.Row():
            with gr.Column():
                track_video_input = gr.Video(label="Input Video")
                preview_btn       = gr.Button("Step 1 — Preview", variant="secondary")
                track_preview_out = gr.Image(label="Frame 0 — pick a player number")
                preview_status    = gr.Textbox(label="Status")
                player_slider     = gr.Slider(
                    minimum=0, maximum=0, step=1, value=0,
                    label="Player index", visible=False
                )
                track_btn = gr.Button("Step 2 — Track", variant="primary")
            with gr.Column():
                track_video_output = gr.Video(label="Output Video")
                track_status       = gr.Textbox(label="Status")

        preview_btn.click(
            preview_first_frame,
            inputs=track_video_input,
            outputs=[track_preview_out, preview_status, player_slider]
        )
        track_btn.click(
            run_track_selected,
            inputs=player_slider,
            outputs=[track_video_output, track_status]
        )

    # ── Tab 4: Export .obj ───────────────────────────────────────────────────
    with gr.Tab("Export .obj"):
        gr.Markdown("Upload an image to export the reconstructed 3D mesh as a `.obj` file.")
        with gr.Row():
            with gr.Column():
                obj_input  = gr.Image(label="Input Image", type="numpy")
                obj_btn    = gr.Button("Export", variant="primary")
            with gr.Column():
                obj_output = gr.File(label="Download .obj")
                obj_status = gr.Textbox(label="Status")
        obj_btn.click(run_save_obj, inputs=obj_input, outputs=[obj_output, obj_status])

demo.launch(share=True)

# IPyWidget

In [4]:
import os
import io
import cv2
import tempfile
import numpy as np
import ipywidgets as widgets
from PIL import Image as PILImage
from IPython.display import display, clear_output

# ------------------------------------------------------------
# CONFIG: set your video filename here (same folder as notebook)
# ------------------------------------------------------------
DEFAULT_VIDEO = "test_data/match.mp4"   # <-- change this to your real filename

# ------------------------------------------------------------
# Helper: numpy RGB image -> widget image
# ------------------------------------------------------------
def np_to_widget(img_rgb: np.ndarray, width=720):
    pil = PILImage.fromarray(img_rgb.astype(np.uint8))
    buf = io.BytesIO()
    pil.save(buf, format="PNG")
    return widgets.Image(value=buf.getvalue(), format="png", width=width)

# ------------------------------------------------------------
# Shared state
# ------------------------------------------------------------
_track_state = {
    "video_path": None,
    "detections": None,
}

# ------------------------------------------------------------
# UI
# ------------------------------------------------------------
video_name = widgets.Text(
    value=DEFAULT_VIDEO,
    description="Video:",
    layout=widgets.Layout(width="500px")
)

preview_btn = widgets.Button(
    description="Preview Players",
    button_style="warning"
)

player_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Player:",
    disabled=True,
    layout=widgets.Layout(width="600px")
)

track_btn = widgets.Button(
    description="Track Selected Player",
    button_style="primary",
    disabled=True
)

out_preview = widgets.Output()
out_result = widgets.Output()
out_status = widgets.Output()

def show_status(msg):
    with out_status:
        clear_output(wait=True)
        print(msg)

# ------------------------------------------------------------
# Step 1: Read first frame + detect players
# ------------------------------------------------------------
def on_preview(_):
    filename = video_name.value.strip()

    if not filename:
        show_status("Please enter a video filename.")
        return

    video_path = os.path.abspath(filename)

    if not os.path.exists(video_path):
        show_status(f"Video not found: {video_path}")
        return

    show_status("Opening video file...")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        show_status(f"OpenCV could not open video:\n{video_path}")
        return

    show_status("Reading first frame...")

    ret, frame_bgr = cap.read()
    cap.release()

    if not ret or frame_bgr is None:
        show_status("Could not read the first frame.")
        return

    show_status(f"First frame loaded: shape={frame_bgr.shape}")

    # OPTIONAL: show raw first frame immediately so we know video loading works
    with out_preview:
        clear_output(wait=True)
        display(np_to_widget(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)))

    show_status("Running person detection on first frame...")

    try:
        boxes = estimator.detector.run_human_detection(
            frame_bgr,
            det_cat_id=0,
            bbox_thr=0.5,
            nms_thr=0.3,
            default_to_full_image=False,
        )
    except Exception as e:
        show_status(f"Detection crashed:\n{type(e).__name__}: {e}")
        return

    show_status(f"Detection finished. Found {len(boxes)} player(s).")

    preview = frame_bgr.copy()

    for idx, box in enumerate(boxes):
        x1, y1, x2, y2 = box[:4].astype(int)
        cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            preview, str(idx),
            (x1 + 5, y1 + 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    _track_state["video_path"] = video_path
    _track_state["detections"] = boxes

    n = len(boxes)
    player_slider.max = max(n - 1, 0)
    player_slider.value = 0
    player_slider.disabled = (n == 0)
    track_btn.disabled = (n == 0)

    with out_preview:
        clear_output(wait=True)
        display(np_to_widget(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)))

    if n == 0:
        show_status("No players detected in first frame.")
    else:
        show_status(f"Detected {n} player(s). Select one and click Track.")

# ------------------------------------------------------------
# Step 2: Track selected player
# ------------------------------------------------------------
def on_track(_):
    video_path = _track_state["video_path"]
    boxes = _track_state["detections"]

    if video_path is None or boxes is None or len(boxes) == 0:
        show_status("Please run Preview Players first.")
        return

    selected_idx = player_slider.value
    selected_bbox = boxes[selected_idx][:4]

    base, ext = os.path.splitext(video_path)
    out_path = f"{base}_tracked{ext}"

    show_status(f"Tracking player {selected_idx}... this may take a while.")

    # Uses your existing notebook function
    _process_video_track_with_bbox(video_path, selected_bbox, out_path)

    with out_result:
        clear_output(wait=True)
        print("Tracking complete.")
        print(f"Saved video: {out_path}")

    show_status("Done.")

preview_btn.on_click(on_preview)
track_btn.on_click(on_track)

display(widgets.VBox([
    widgets.HTML("<h3>Track a Player from Local Video</h3>"),
    video_name,
    preview_btn,
    out_preview,
    player_slider,
    track_btn,
    out_result,
    out_status
]))

In [ ]:
!pip install --quiet fastapi uvicorn[standard] python-multipart

In [ ]:
import subprocess, threading, time, os

# Copy main.py and static/ into the working directory if running from notebook dir
# (adjust paths if your structure differs)
os.makedirs("static", exist_ok=True)

def _run_server():
    subprocess.run([
        "uvicorn", "main:app",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--reload",
    ])

thread = threading.Thread(target=_run_server, daemon=True)
thread.start()
time.sleep(2)
print("Server running at http://localhost:8000")
print("If on a remote server, forward the port with:")
print("  ssh -L 8000:localhost:8000 user@your-server")
print("Then open http://localhost:8000 in your browser.")
